# 7회차 · 마지막 시간 : 부품을 조립해 나만의 AI 서비스 만들기(최종 스튜디오)

이 노트북은 완성된 챗봇을 써 보는 노트북이 아님. 1 · 2 · 5 · 6 · 7회차에 배운 것을 **부품 하나하나의 코드**로 다시 꺼내고, 값을 바꿔 보며 확인한 뒤, 그 코드를 이어 붙여 **내가 조립한 챗봇**과 **다른 프로그램이 부를 수 있는 API** 를 만듦. 마지막으로 같은 부품으로 다른 분야의 응용을 하나 만들어 봄.

**시작 전에 꼭 할 것** : 7회차 학습 노트북(`s07_my_lora.ipynb`)의 커널을 끔(위쪽 메뉴 Kernel → Shut Down Kernel). 같은 GPU 에 모델이 두 번 올라가면 메모리가 모자람. `lab.gpu()` 계기판에 '이 노트북 밖에서 GPU 메모리를 쓰고 있음'이 보이면 아직 꺼지지 않은 것임.

| 순서 | 하는 일 | 핵심 코드 | 배운 회차 |
|---|---|---|---|
| S1 | LLM 으로 답 만들기 : 채팅 템플릿 → 토큰 → 생성 | `tok.apply_chat_template` · `model.generate` | 2회차(디코딩) · 7회차(Adapter) |
| S2 | 검색해서 문서 붙이기(RAG) | 코사인 유사도 · `lab.search` | 6회차 |
| S3 | 대화 상태를 JSON 으로 정리 | 지시문 · `json.loads` · 다시 시도 | 5회차 |
| S4 | 상태에 맞는 그림 | `pipe(prompt, num_inference_steps, guidance_scale)` | 오늘 처음(확산 모델) |
| S5 | 답을 소리로 읽기 | `VitsModel` · 숫자 읽기 | 오늘 처음(VITS) |
| S6 | 소리를 글로 받아쓰기 | Whisper `pipeline` | 1회차(인코더-디코더) · 2회차(Transformer) |
| S7 | 조립 : 내 챗봇 함수 `my_bot` | 부품 함수 이어 붙이기 · 시간 재기 | 전체 |
| S8 | 출력 차이 실험 : 검색 · Adapter 를 켜고 끄기 | 같은 질문 네 구성 | 6 · 7회차 |
| S9 | 웹 화면과 API | FastAPI 경로 함수 | 9 · 10회차로 이어짐 |
| S10 | 응용 하나 만들기 : 회화 튜터 · 그림책 · 음성 메모 요약 · 안내 키오스크 | 부품 다시 조합 | 전체 |
| S11 | 추가 학습(LLM) : 응용에 맞는 Adapter | `lab.train` | 7회차 |
| S12 | 한계 정리 · 나만의 솔루션 기획 · 결과 묶기 | - | - |

- 셀마다 맨 위의 **대문자 이름**(예 : `TEMPERATURE = 0.7`)이 바꿔 볼 값임. 값을 바꾸고 다시 실행하기 전에 결과를 먼저 예측해 적음
- 셀 안의 함수는 내가 고쳐 써도 됨. S7 의 `my_bot` 은 S1~S6 에서 만든 함수를 부르므로, 앞 셀의 함수를 고치면 조립한 챗봇도 바뀜
- `lab.xxx` 는 모델을 내려받고 GPU 에 올리는 일만 맡음. 실제 동작은 이 노트북의 코드가 함. 궁금하면 `lab.code("함수 이름")` 으로 lab 쪽 코드도 볼 수 있음

※ 11/23 밤 강사가 A100 80GB 서버에서 처음부터 끝까지 실행해 본 노트북임(오류 없음). S0 솔루션 올리기 약 3분, 그림 · 읽어 주기 · 받아쓰기 모델을 처음 올리는 데 각각 38초 · 7초 · 14초, S7 한 턴 약 3초, S11 추가 학습 약 3분이었음. 수업 전에 서버마다 모델을 미리 받아 두었음. 미리 받지 않은 서버라면 내려받기로 1~2분씩 더 걸림

## S0. 시작
아래 셀은 7회차 학습 노트북과 같음. 6 · 7회차에 저장한 솔루션(모델 · 문서 · Adapter)이 올라옴.

In [ ]:
import sys, pathlib
root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "lab.py").exists())
sys.path.insert(0, str(root / "src"))
import lab
lab.start(7)

In [ ]:
lab.load_solution()
lab.gpu()

## S1. LLM 으로 답 만들기(2회차 디코딩 · 7회차 Adapter)
LLM 은 '다음 토큰 맞히기'를 반복해 글을 만듦. 대화는 **채팅 템플릿**으로 한 줄의 글이 되고, 토크나이저가 토큰 번호로 바꾸고, `generate` 가 토큰을 하나씩 이어 붙임. 2회차에서 본 디코딩 설정이 S1 셀에서 그대로 쓰임.

| 값 | 뜻 | 바꿔 볼 것 |
|---|---|---|
| `TEMPERATURE` | 0 이면 늘 가장 확률 높은 토큰(같은 질문 = 같은 답). 클수록 다양하지만 말이 흩어짐 | 0 → 0.7 → 1.5, 같은 질문을 세 번씩 |
| `TOP_P` | 확률이 높은 순서로 더한 값이 TOP_P 가 될 때까지만 후보로 둠 | 0.5 대 0.95 |
| `MAX_NEW_TOKENS` | 답의 최대 길이(토큰). 넘으면 문장 중간에서 끊김 | 30 대 300 |
| `ADAPTER` | 7회차에 학습한 Adapter. `None` 이면 학습 전 모델 | 저장한 Adapter 대 `None` |

코드에서 볼 곳 : `llm_parts` 가 돌려주는 `ctx` 는 Adapter 를 잠시 끄는 장치임. Adapter 를 끈 답을 만들 때 `with ctx:` 안에서 생성함.

In [ ]:
# S1. LLM 부품 : 채팅 템플릿 → 토큰 → 생성 → 글자
import torch
ADAPTER = lab.cfg().get("adapter")      # 10단계에서 저장한 Adapter. None 이면 학습 전 모델
TEMPERATURE = 0.7                       # 0 이면 같은 질문에 늘 같은 답
TOP_P = 0.9
MAX_NEW_TOKENS = 200
REPETITION_PENALTY = 1.1                # 1 보다 크면 같은 말을 되풀이하지 않게 누름

def generate(messages, adapter=ADAPTER, temperature=TEMPERATURE, top_p=TOP_P, max_new_tokens=MAX_NEW_TOKENS):
    tok, model, ctx = lab.llm_parts(adapter)                                   # 올려 둔 본체(+ Adapter)를 꺼냄
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    ids = tok(text, return_tensors="pt").to(model.device)
    sample = temperature > 0
    with ctx, torch.no_grad():
        out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=sample,
                             temperature=temperature if sample else None, top_p=top_p if sample else None,
                             repetition_penalty=REPETITION_PENALTY, pad_token_id=tok.pad_token_id)
    new = out[0, ids["input_ids"].shape[1]:]                                   # 입력 뒤에 새로 붙은 토큰만
    return tok.decode(new, skip_special_tokens=True).strip()

msgs = [{"role": "user", "content": "주말에 뭐 하면 좋을까?"}]
tok, _, _ = lab.llm_parts(ADAPTER)
print("[모델에 실제로 들어가는 글]")
print(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))
print("[토큰 수]", len(tok(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))["input_ids"]))
print()
print(f"[Adapter {ADAPTER}]", generate(msgs))
print()
print("[Adapter 끔]", generate(msgs, adapter=None))

In [ ]:
# S1 실험 : 온도에 따라 같은 질문의 답이 얼마나 달라지나(값을 바꿔 다시 실행)
for t in (0, 0.7, 1.5):
    print(f"--- temperature {t}")
    for _ in range(2):
        print(" ·", generate(msgs, temperature=t, max_new_tokens=60).replace("\n", " ")[:120])

<details><summary>확인 질문 : temperature 0 으로 두 번 물으면 답이 같은가. 1.5 에서는 무엇이 달라지나</summary>

0 은 매번 가장 확률이 높은 토큰만 고르므로 같은 답이 나옴(2회차 그리디 디코딩). 1.5 는 확률이 낮은 토큰도 자주 뽑혀 표현이 다양해지는 대신 문장이 어색해지거나 주제를 벗어나기 쉬움. 서비스에서는 사실을 답할 때 낮게, 아이디어를 낼 때 조금 높게 둠
</details>

## S2. 검색해서 문서 붙이기(6회차 RAG)
6회차의 검색을 코드로 다시 봄. 질문과 청크를 임베딩(길이 1인 벡터)으로 바꾸면 **내적이 곧 코사인 유사도**임. `lab.search` 는 여기에 키워드 점수(BM25)를 섞은 하이브리드 검색임. 찾은 청크를 지시문과 함께 붙여 S1 의 `generate` 에 넘기면 RAG 가 됨.

| 값 | 뜻 | 바꿔 볼 것 |
|---|---|---|
| `TOP_K` | 붙일 청크 수. 많으면 근거가 늘지만 입력이 길어지고 관련 없는 청크도 섞임 | 1 대 4 대 8 |
| `KW` | 키워드 점수의 비중(0 = 의미만, 1 = 키워드만) | 0 대 0.3 대 1 |
| `RAG_SYSTEM` | 문서를 어떻게 쓰라고 시키는 지시문 | '모르면 모른다고' 문장을 뺐을 때 |

In [ ]:
# S2. 검색 부품 : 코사인 유사도를 직접 계산해 보고, 하이브리드 검색으로 문서를 붙임
import numpy as np
TOP_K = 4
KW = 0.3
RAG_SYSTEM = "아래 [참고 문서]의 내용을 근거로 답한다. 문서에 없는 내용은 지어내지 않고 모른다고 답한다."

q = "동아리 회비는 얼마고 언제 내?"                     # 내 문서 주제로 바꿈
qv = lab.embed([q])[0]                                  # (차원,) · 길이 1
cs = lab.chunks()
sims = lab.S.emb @ qv                                   # 모든 청크와의 코사인 유사도를 한 번에(행렬 × 벡터)
print("[의미 검색만 : 코사인 유사도 상위 3]")
for i in np.argsort(-sims)[:3]:
    print(f"  {sims[i]:.3f}  {cs[i]['title'][:16]} | {cs[i]['text'][:60]}")

def retrieve(q, k=TOP_K, kw=KW):
    return lab.search(q, k=k, kw=kw)                    # 의미 점수와 키워드 점수를 섞은 순위

def with_context(q, hits, history=()):
    ctx = "\n\n".join(f"<{h['title']}>\n{h['text']}" for h in hits)
    return [{"role": "system", "content": RAG_SYSTEM}, *history,
            {"role": "user", "content": f"[참고 문서]\n{ctx}\n\n[질문]\n{q}"}]

hits = retrieve(q)
print("\n[하이브리드 검색 상위]", [(h["rank"], h["title"][:12], round(h["score"], 2)) for h in hits])
print("\n[문서 붙여 답함]", generate(with_context(q, hits), temperature=0))
print("\n[문서 없이 답함]", generate([{"role": "user", "content": q}], temperature=0))

<details><summary>확인 질문 : 내 문서에 없는 것을 물으면 두 답이 어떻게 다른가</summary>

문서를 붙인 답은 지시문대로 '문서에 없다'고 답하는 경우가 많고, 문서 없이 답하면 그럴듯하게 지어내기 쉬움(6회차 '문서에 없는 질문'). 다만 지시문을 빼거나 관련 없는 청크가 붙으면 문서를 붙여도 지어냄. `RAG_SYSTEM` 을 바꿔 확인해 봄
</details>

## S3. 대화 상태를 JSON 으로 정리(5회차 형식 지정 출력)
그림과 소리를 붙이려면 '지금 대화가 어떤 분위기인가'를 프로그램이 읽을 수 있는 형태로 받아야 함. 5회차처럼 지시문으로 **형식**을 정하고, 받은 글을 `json.loads` 로 읽고, 실패하면 다시 시킴. 같은 LLM 을 쓰되 Adapter 는 끔(말투가 섞이면 JSON 이 깨지기 쉬움).

| 값 | 뜻 | 바꿔 볼 것 |
|---|---|---|
| `MOODS` | 고를 수 있는 분위기 목록 | 분위기를 셋으로 줄이거나 새로 추가 |
| `STATE_SYSTEM` | 형식을 정하는 지시문 | 예시 한 건을 넣었을 때(5회차 예시 효과) |
| `RETRY` | JSON 이 깨졌을 때 다시 시킬 횟수 | 0 대 2 |

In [ ]:
# S3. 상태 정리 부품 : 지시문으로 형식을 정하고, 읽고, 실패하면 다시 시킴
import json, re
MOODS = ["기쁨", "차분", "걱정", "놀람", "진지", "장난"]
RETRY = 1
STATE_SYSTEM = (
    "너는 챗봇 화면을 꾸미는 연출가다. 사용자 질문과 챗봇의 답을 읽고 지금 상태를 JSON 하나로만 쓴다.\n"
    f'형식 : {{"mood": "{", ".join(MOODS)} 가운데 하나", "scene": "장면을 한국어 한 문장으로", '
    '"prompt": "그 장면을 그릴 영어 묘사, 20단어 이내"}\n'
    "JSON 밖에는 아무것도 쓰지 않는다.")

def decide_state(q, answer, retry=RETRY):
    msgs = [{"role": "system", "content": STATE_SYSTEM}, {"role": "user", "content": f"[질문]\n{q}\n\n[챗봇의 답]\n{answer[:500]}"}]
    for i in range(retry + 1):
        raw = generate(msgs, adapter=None, temperature=0 if i == 0 else 0.7, max_new_tokens=160)
        m = re.search(r"\{.*\}", raw, re.S)
        try:
            st = json.loads(m.group(0))
            st["mood"] = next((x for x in MOODS if x in str(st.get("mood"))), "차분")
            st["prompt"] = re.sub(r"[^A-Za-z0-9 ,.'-]", " ", str(st.get("prompt", "")))   # 그림 모델은 영어만 읽음
            return st, raw, i
        except Exception:
            continue
    return {"mood": "차분", "scene": "-", "prompt": ""}, raw, retry

for qq, aa in [("나 오늘 시험 합격했어!", "축하하오! 그동안 애쓴 보람이 있구려."), ("비 오는데 우산을 잃어버렸어", "저런, 근처 편의점에서 하나 사는 것이 어떻소.")]:
    st, raw, tries = decide_state(qq, aa)
    print(qq, "→", st, f"(다시 시킨 횟수 {tries})")
print("\n[모델이 실제로 낸 글]", raw)

## S4. 상태에 맞는 그림(오늘 처음 : 확산 모델)
그림 모델은 잡음에서 시작해 묘사에 맞게 잡음을 조금씩 걷어 내며 그림을 만듦(확산). 보통 수십 번 반복하지만 sd-turbo 는 그 과정을 1~4번으로 줄이도록 다시 학습한 모델이라 1초 안쪽에 그림. 묘사의 앞부분을 읽는 것이 CLIP 이라는 Transformer 인코더임(2회차 인코더와 같은 구조).

| 값 | 뜻 | 바꿔 볼 것 |
|---|---|---|
| `STEPS` | 잡음을 걷어 내는 횟수 | 1 · 2 · 4 |
| `SEED` | 처음 잡음을 정하는 값. 같으면 같은 그림 | 같은 묘사로 seed 만 바꿈 |
| `CHARACTER` · `STYLE` | 모든 그림에 붙일 캐릭터와 화풍 | 캐릭터를 바꾸면 그림 속 인물이 바뀜 |
| `MOOD_LOOK` | 분위기마다 붙일 표정 묘사 | 표정 묘사를 지웠을 때 |

sd-turbo 에는 안전 필터가 없음. 수업에서는 lab 이 묘사에 걸리는 낱말이 있으면 다른 그림으로 바꿈. 사용 조건은 모델 카드의 라이선스를 따름.

In [ ]:
# S4. 그림 부품 : 묘사 → 그림(sd-turbo)
import torch
STEPS = 1
SEED = 7
SIZE = 512
CHARACTER = "a friendly Joseon-era scholar in white hanbok and black gat hat"   # 내 Adapter 말투에 맞게 바꿈
STYLE = "storybook illustration, soft pastel colors, simple background"
MOOD_LOOK = {"기쁨": "smiling brightly", "차분": "calm gentle smile", "걱정": "worried expression",
             "놀람": "surprised wide eyes", "진지": "serious thoughtful face", "장난": "playful wink"}

pipe = lab.image_load()["pipe"]                          # 처음 한 번 설치 · 내려받기(1~2분)

def draw(scene_prompt, mood="차분", steps=STEPS, seed=SEED):
    prompt = ", ".join(x for x in (CHARACTER, MOOD_LOOK.get(mood, ""), scene_prompt, STYLE) if x)
    g = torch.Generator("cpu").manual_seed(seed + MOODS.index(mood) if mood in MOODS else seed)   # 분위기마다 seed 를 달리함
    return pipe(prompt=prompt, num_inference_steps=steps, guidance_scale=0.0, height=SIZE, width=SIZE, generator=g).images[0], prompt

img, used = draw("reading a letter under a cherry tree", "기쁨")
print(used)
img.resize((256, 256))

In [ ]:
# S4 실험 : 단계 수와 seed 바꾸기(같은 묘사)
from IPython.display import display
import time
for st in (1, 2, 4):
    t = time.time(); im, _ = draw("watching stars on a rooftop", "차분", steps=st)
    print(f"steps {st} · {time.time() - t:.2f}초"); display(im.resize((192, 192)))

## S5. 답을 소리로 읽기(오늘 처음 : VITS 음성 합성)
글을 소리로 바꾸는 모델(MMS-TTS, 약 150MB). 한국어 모델은 한글을 로마자로 바꿔 넣고, **숫자는 읽지 못하므로** 먼저 한글로 바꿈(`lab.read_numbers`). 작은 모델이라 기계 소리가 나고 억양이 단조로움. 로컬 소형 모델의 한계임.

| 값 | 뜻 | 바꿔 볼 것 |
|---|---|---|
| `RATE` | 말 빠르기(1.0 기본) | 0.8 대 1.3 |
| `NOISE` | 억양 변화의 크기(클수록 덜 단조롭지만 불안정) | 0.3 대 0.667 대 1.0 |
| 숫자 바꾸기 | `read_numbers` 를 쓰는가 | 숫자를 그대로 넣었을 때 어떻게 읽는가 |

In [ ]:
# S5. 읽어 주기 부품 : 글 → (숫자 읽기 · 로마자) → 소리
from IPython.display import Audio
RATE = 1.0
NOISE = 0.667
tts_tok, tts_model, romanize = lab.tts_load("ko")

def say(text, rate=RATE, noise=NOISE, fix_numbers=True):
    src = lab.read_numbers(text) if fix_numbers else text
    tts_model.speaking_rate, tts_model.noise_scale = rate, noise
    ids = tts_tok(romanize(src[:300]), return_tensors="pt").to(tts_model.device)
    with torch.no_grad():
        wave = tts_model(**ids).waveform[0].float().cpu().numpy()
    return lab._wav(wave, tts_model.config.sampling_rate), src

text = "동아리 회비는 한 학기에 2만 원이고, 망원경은 3일까지 빌릴 수 있습니다."
wav, read_as = say(text)
print("읽은 글 :", read_as)
print("로마자 :", romanize(read_as)[:80])
Audio(wav, autoplay=False)

In [ ]:
# S5 실험 : 숫자를 그대로 넣으면?(fix_numbers=False) · 빠르기
wav2, _ = say(text, fix_numbers=False)
display(Audio(wav2))
wav3, _ = say(text, rate=1.3)
display(Audio(wav3))

## S6. 소리를 글로 받아쓰기(1회차 인코더-디코더 · 2회차 Transformer)
Whisper 는 1회차에서 배운 **인코더-디코더** 구조임. 인코더가 소리(멜 스펙트로그램)를 읽어 벡터로 만들고, 디코더가 그 벡터를 보며 글자 토큰을 하나씩 생성함(번역 모델과 같은 방식). S6 셀에서는 S5 에서 만든 소리를 다시 받아써서 읽기 · 받아쓰기가 서로 맞는지 봄.

| 값 | 뜻 | 바꿔 볼 것 |
|---|---|---|
| `ASR_SIZE` | `big`(large-v3-turbo, 약 1.6GB) · `small`(약 0.5GB) | 같은 소리를 둘 다로 |
| `LANG` | 언어를 정해 줄지(`None` 이면 모델이 알아냄) | `"korean"` 대 `None` |
| 잡음 | 소리에 잡음을 섞었을 때 | `NOISE_LEVEL` 0 대 0.05 |

In [ ]:
# S6. 받아쓰기 부품 : 소리 → 글(Whisper)
import io, wave
ASR_SIZE = "big" if lab.gpu(show=False)["total"] > 30 else "small"   # big : large-v3-turbo(약 1.6GB) · small : 약 0.5GB. 24GB 서버는 small
LANG = "korean"
NOISE_LEVEL = 0.0
asr = lab.asr_load(ASR_SIZE)

def hear(audio, lang=LANG):
    """audio : wav 바이트 · 파일 경로 · (배열, 표본 빈도)"""
    if isinstance(audio, (bytes, bytearray)):            # wav 바이트 → 배열
        w = wave.open(io.BytesIO(audio)); sr = w.getframerate()
        audio = (np.frombuffer(w.readframes(w.getnframes()), dtype=np.int16).astype(np.float32) / 32768, sr)
    if isinstance(audio, tuple):
        y, sr = audio
        y = y + NOISE_LEVEL * np.random.randn(len(y)).astype(np.float32)
        if sr != 16000:                                  # Whisper 는 초당 16,000개 표본을 받음
            from scipy.signal import resample_poly
            y = resample_poly(y, 16000, sr)
        audio = {"raw": y.astype(np.float32), "sampling_rate": 16000}
    r = asr(audio, generate_kwargs={"task": "transcribe", **({"language": lang} if lang else {})})
    return r["text"].strip()

print("원래 글  :", text)
print("받아쓴 글 :", hear(wav))

<details><summary>확인 질문 : 받아쓴 글이 원래 글과 다르다. 읽기와 받아쓰기 가운데 어느 쪽의 문제인지 어떻게 가르나</summary>

부품을 하나씩 바꿔 봄. 사람이 녹음한 소리(웹 화면의 마이크)를 받아써서 맞으면 읽기(TTS)의 발음 문제이고, 그래도 틀리면 받아쓰기(ASR)의 문제임. 숫자 · 고유 명사(동아리 이름)가 자주 틀림. 부품을 이어 붙이면 앞 부품의 실수가 뒤로 그대로 넘어감
</details>

## S7. 조립 : 내 챗봇 함수 `my_bot`
S1~S6 에서 만든 함수를 이어 붙임. 순서는 **받아쓰기 → 검색 → 답 → 상태 정리 → 그림 → 읽어 주기**. 부품마다 걸린 시간을 재서 어디가 느린지 봄. 이 함수가 S9 의 웹 화면과 API 가 부르는 바로 그 함수임.

고쳐 볼 것
- 순서 바꾸기 : 상태를 먼저 정하고 그 분위기를 답의 지시문에 넣으면?
- 부품 빼기 : `draw=False` · `voice=False` 로 시간이 얼마나 줄어드나
- 기억 붙이기 : `history` 에 앞의 대화를 넣어 '그거 다시 말해 줘'가 통하는지

In [ ]:
# S7. 조립 : 부품 함수를 이어 붙인 내 챗봇
import time
def my_bot(q=None, audio=None, rag=True, adapter=ADAPTER, draw_on=True, voice=True, history=()):
    t, out = {}, {}
    if audio is not None:                                            # ① 받아쓰기
        t0 = time.time(); q = hear(audio); t["받아쓰기"] = round(time.time() - t0, 2)
    t0 = time.time()
    hits = retrieve(q) if rag else []                                # ② 검색
    t["검색"] = round(time.time() - t0, 2)
    msgs = with_context(q, hits, list(history)) if hits else [*history, {"role": "user", "content": q}]
    t0 = time.time(); answer = generate(msgs, adapter=adapter, temperature=0.3); t["답"] = round(time.time() - t0, 2)   # ③ 답
    t0 = time.time(); st, _, _ = decide_state(q, answer); t["상태"] = round(time.time() - t0, 2)                       # ④ 상태
    img = None
    if draw_on:
        t0 = time.time(); img, _ = draw(st["prompt"], st["mood"]); t["그림"] = round(time.time() - t0, 2)              # ⑤ 그림
    wav = None
    if voice:
        t0 = time.time(); wav, _ = say(answer); t["소리"] = round(time.time() - t0, 2)                                # ⑥ 읽어 주기
    return {"q": q, "answer": answer, "sources": [h["title"] for h in hits], "state": st, "image": img, "audio": wav, "times": t}

r = my_bot("동아리 회비는 얼마고 언제 내?")                          # 내 문서 주제로 바꿈
lab.show_turn(r)

In [ ]:
# S7 : 소리로 묻기(S5 로 만든 질문 소리를 그대로 넣음. 웹 화면에서는 마이크로)
q_wav, _ = say("망원경은 며칠까지 빌릴 수 있어?")
r = my_bot(audio=q_wav)
lab.show_turn(r)

## S8. 출력 차이 실험 : 검색과 Adapter 를 켜고 끄기(6 · 7회차 정리)
같은 질문을 네 구성으로 물어 **무엇이 바뀌는지** 표로 봄. 그림 · 소리는 끄고 글만 비교함. 7회차 학습 노트북 9-1(체감 실험실)에서 본 'LoRA 로 지식 넣기'의 결과와 함께 판단함.

| 구성 | 기대하는 것 |
|---|---|
| 둘 다 끔 | 기본 말투, 내 문서의 사실은 모름(지어낼 수 있음) |
| 검색만 | 사실은 맞음, 말투는 기본 |
| Adapter 만 | 말투는 목표대로, 사실은 모름 |
| 둘 다 | 사실과 말투 모두. 문서를 붙이면 말투가 약해지는지 확인 |

In [ ]:
# S8. 네 구성 비교표(질문을 바꿔 봄)
QS = ["동아리 회비는 얼마고 언제 내?", "정기 모임은 언제 해?"]
rows = []
for qq in QS:
    for rag_on, ad in [(False, None), (True, None), (False, ADAPTER), (True, ADAPTER)]:
        r = my_bot(qq, rag=rag_on, adapter=ad, draw_on=False, voice=False)
        rows.append({"질문": qq[:14], "검색": "켬" if rag_on else "끔", "Adapter": ad or "끔", "답": r["answer"][:90].replace("\n", " "),
                     "말투": lab._style(r["answer"], lab.goal()), "시간": sum(r["times"].values())})
import pandas as pd
pd.DataFrame(rows)

## S9. 웹 화면과 API(9 · 10회차로 이어짐)
`my_bot` 을 다른 프로그램이 부를 수 있게 **API** 로 내보냄. FastAPI 는 함수 위에 `@app.post("/api/chat")` 처럼 경로를 붙이면 그 주소로 들어온 요청을 함수로 넘겨 줌. 요청의 모양은 `pydantic` 클래스로 정하고, 이 모양이 `/docs` 의 자동 문서가 됨. 웹 화면(`src/studio_page.html`)은 이 API 를 부르는 페이지일 뿐이므로, 같은 API 를 다른 앱 · 다른 언어에서도 부를 수 있음.

- 여러 사람이 동시에 부르면 GPU 계산이 엉킬 수 있어 `LOCK` 으로 한 번에 하나씩 처리함
- 띄운 주소는 누구나 열 수 있음. 수업이 끝나면 `lab.stop_api()` 로 끔
- 8 · 9 · 10회차에는 이 구조를 Agent 와 서비스로 키움

In [ ]:
# S9. API : 경로 함수 네 개(고쳐서 경로를 더 만들어도 됨)
import base64, io, threading
from fastapi import FastAPI
from fastapi.responses import HTMLResponse
from pydantic import BaseModel

app = FastAPI(title="나만의 챗봇 API", description="7회차 최종 스튜디오에서 조립한 my_bot 을 부르는 API")
LOCK = threading.Lock()

def b64_image(img):
    if img is None: return None
    buf = io.BytesIO(); img.convert("RGB").save(buf, format="JPEG", quality=85); return base64.b64encode(buf.getvalue()).decode()

class ChatIn(BaseModel):
    q: str
    rag: bool = True
    adapter: str | None = ADAPTER
    draw: bool = True
    voice: bool = True
    history: list = []

@app.post("/api/chat")
def api_chat(x: ChatIn):
    with LOCK:
        r = my_bot(x.q, rag=x.rag, adapter=x.adapter, draw_on=x.draw, voice=x.voice, history=x.history[-4:])
    return {"answer": r["answer"], "sources": r["sources"], "state": r["state"], "times": r["times"],
            "image": b64_image(r["image"]), "audio": base64.b64encode(r["audio"]).decode() if r["audio"] else None}

class AudioIn(BaseModel):
    audio_b64: str
    fmt: str = "webm"

@app.post("/api/asr")
def api_asr(x: AudioIn):
    path = lab.MY / f"rec.{x.fmt}"
    path.write_bytes(base64.b64decode(x.audio_b64))      # 브라우저 녹음(webm)은 ffmpeg 로 읽음
    with LOCK:
        return {"text": asr(str(path), generate_kwargs={"task": "transcribe", "language": LANG})["text"].strip()}

@app.get("/api/info")
def api_info():
    return {"model": lab.cfg()["model"], "adapter": ADAPTER, "adapters": [x["이름"] for x in lab.runs(show=False)], "docs": len(lab.docs(show=False))}

@app.get("/", response_class=HTMLResponse)
def page():
    return (lab.SRC / "studio_page.html").read_text(encoding="utf-8")

url = lab.run_api(app)                                   # 이 서버의 8000번으로 띄움(노트북의 모델을 그대로 씀)

In [ ]:
# S9 : 다른 프로그램에서 API 부르기(파이썬 requests). 다른 PC 에서는 local 대신 위에 나온 url 을 씀
import requests
local = "http://localhost:8000"
res = requests.post(local + "/api/chat", json={"q": "정기 모임은 언제 해?", "draw": False, "voice": False}, timeout=120).json()
print(res["answer"])
print(res["times"])
# 터미널에서는 : curl -X POST <주소>/api/chat -H "Content-Type: application/json" -d '{"q":"정기 모임은 언제 해?","draw":false,"voice":false}'

## S10. 응용 하나 만들기(같은 부품, 다른 분야)
아래 네 가지 가운데 하나를 골라 실행하고, 값과 지시문을 내 분야에 맞게 고침. 모두 S1~S6 의 함수만 다시 조합한 것임. 시간이 남으면 두 번째를 함.

| 응용 | 부품 조합 | 고쳐 볼 것 |
|---|---|---|
| A. 영어 회화 튜터 | 받아쓰기(영어) → 문장 고쳐 주기(LLM) → 영어로 읽어 주기 | 고쳐 주는 지시문, 수준(초급 · 중급) |
| B. 그림책 만들기 | 줄거리 JSON(LLM) → 장면마다 그림 → 낭독 → HTML 한 장 | 장 수, 화풍, 등장인물 |
| C. 음성 메모 요약 | 받아쓰기 → 세 줄 요약(5회차 형식) → 읽어 주기 | 요약 형식(할 일 목록, 회의록) |
| D. 안내 키오스크 | 소리로 묻기 → 내 문서 검색 → 짧게 답하고 읽어 주기 | 답 길이, 말투(Adapter), 문서 |

In [ ]:
# S10-A. 영어 회화 튜터 : 영어로 말하면 문법을 고쳐 주고, 고친 문장을 영어로 읽어 줌
TUTOR_SYSTEM = "You are a friendly English tutor for Korean university students. Correct the student's sentence, then explain the fix in one short Korean sentence. Format: Corrected: ... / 설명: ..."
en_tok, en_model, _ = lab.tts_load("en")

def say_en(text):
    ids = en_tok(text[:300], return_tensors="pt").to(en_model.device)
    with torch.no_grad():
        w = en_model(**ids).waveform[0].float().cpu().numpy()
    return lab._wav(w, en_model.config.sampling_rate)

student_wav = say_en("Yesterday I go to the library and borrow two book.")   # 학생 목소리 대신(웹 화면에서는 마이크)
heard = hear(student_wav, lang="english")
fix = generate([{"role": "system", "content": TUTOR_SYSTEM}, {"role": "user", "content": heard}], adapter=None, temperature=0)
print("들은 문장 :", heard); print(fix)
corrected = fix.split("Corrected:")[-1].split("/")[0].strip() if "Corrected:" in fix else fix
Audio(say_en(corrected))

In [ ]:
# S10-B. 그림책 만들기 : 줄거리를 JSON 으로 받고, 장면마다 그림과 낭독을 붙여 HTML 한 장으로 저장
PAGES = 4
TOPIC = "별을 좋아하는 작은 고양이가 동아리 망원경으로 처음 토성을 본 날"
story_raw = generate([{"role": "system", "content": f"어린이 그림책 작가다. JSON 배열 하나로만 답한다. 원소는 {PAGES}개이고 각 원소는 "
                                                    '{"text": "한국어 두 문장", "prompt": "그 장면의 영어 묘사 15단어 이내"} 이다.'},
                      {"role": "user", "content": TOPIC}], adapter=None, temperature=0.7, max_new_tokens=600)
try:
    pages = json.loads(re.search(r"\[.*\]", story_raw, re.S).group(0))
except Exception:                                          # 형식이 깨지면(5회차) 모델이 낸 글을 보고 다시 실행
    print(story_raw); raise
html_parts = []
for i, pg in enumerate(pages[:PAGES]):
    CHARACTER_SAVE, CHARACTER = CHARACTER, "a small curious grey kitten"
    im, _ = draw(pg["prompt"], "기쁨", seed=100 + i); CHARACTER = CHARACTER_SAVE
    w, _ = say(pg["text"])
    html_parts.append(f"<div style='margin:16px 0'><img src='data:image/jpeg;base64,{b64_image(im)}' width=320><p>{pg['text']}</p>"
                      f"<audio controls src='data:audio/wav;base64,{base64.b64encode(w).decode()}'></audio></div>")
out = lab.MY / "storybook.html"
out.write_text("<meta charset='utf-8'><h2>" + TOPIC + "</h2>" + "".join(html_parts), encoding="utf-8")
print("저장 :", out, "(왼쪽 파일 창에서 my/storybook.html 을 오른쪽 단추 → Open in New Browser Tab)")
from IPython.display import HTML
HTML("".join(html_parts[:1]))

In [ ]:
# S10-C. 음성 메모 요약 : 긴 말 → 받아쓰기 → 세 줄 요약 → 읽어 주기
memo_wav, _ = say("오늘 동아리 회의에서 정한 것입니다. 다음 주 토요일에 관측회를 하고, 회비는 이번 달 말까지 걷습니다. 발표는 두 명이 맡고 장비 점검은 금요일에 합니다.")
memo = hear(memo_wav)
summary = generate([{"role": "system", "content": "회의 메모를 정확히 세 줄로 요약한다. 각 줄은 '1)', '2)', '3)'으로 시작한다."},
                    {"role": "user", "content": memo}], adapter=None, temperature=0)
print("받아쓴 메모 :", memo); print(summary)
Audio(say(summary)[0])

In [ ]:
# S10-D. 안내 키오스크 : 짧게 답하게 지시문을 바꾸고 소리로 묻고 답함
RAG_SYSTEM_SAVE = RAG_SYSTEM
RAG_SYSTEM = RAG_SYSTEM_SAVE + " 안내 키오스크이므로 두 문장 이내로 짧게 답한다."
r = my_bot(audio=say("동아리방은 어디에 있어?")[0], draw_on=False)
RAG_SYSTEM = RAG_SYSTEM_SAVE
lab.show_turn(r)

## S11. 추가 학습(LLM) : 응용에 맞는 Adapter
응용에 맞는 말투 · 형식이 있으면 7회차에 한 그대로 Adapter 를 하나 더 학습함. 학습은 LLM 에만 함(그림 · 소리 모델은 그대로 씀). 아래는 '친절한 상담원' 목표로 데이터를 만들고 학습해 S7 의 `my_bot` 에 바로 넣는 예임. 2~4분 걸림.

- 목표 · 데이터 · 학습 코드는 7회차 노트북 1 · 3 · 4단계와 같음. 바뀐 것은 `ADAPTER` 이름뿐임
- 학습이 끝나면 S9 의 웹 화면 Adapter 목록에도 새 이름이 보임(화면을 새로 고침)

In [ ]:
# S11. 추가 학습 : 목표 저장 → 데이터 → 학습 → 내 챗봇에 넣기(7회차 목표와 데이터는 따로 두었다가 되돌림)
import shutil
GOAL = "친절한 상담원"                                     # lab.PRESETS 의 이름 또는 직접 정한 목표
QUANT = "nf4" if lab.gpu(show=False)["total"] < 30 else "bf16"   # 24GB 서버는 본체를 4bit 로(QLoRA)
old_goal = lab.goal()
for f in ("train.jsonl", "valid.jsonl"):
    if (lab.MY / f).exists(): shutil.copy(lab.MY / f, lab.MY / f"7회차_{f}")
p = lab.PRESETS[GOAL]
lab.set_goal(GOAL, p["instruction"], p["markers"], p["examples"], p["rule"])
lab.make_data(n=200, show=False)
new = lab.train(run="kiosk", epochs=2, quant=QUANT)         # 에폭 2 : 4-5 단계별 평가에서 2에폭이면 충분했음
if old_goal: lab.set_goal(old_goal["name"], old_goal["instruction"], old_goal["markers"], old_goal["examples"], old_goal.get("rule", "any"))
for f in ("train.jsonl", "valid.jsonl"):
    if (lab.MY / f"7회차_{f}").exists(): shutil.copy(lab.MY / f"7회차_{f}", lab.MY / f)
ADAPTER = new
lab.show_turn(my_bot("동아리 가입은 어떻게 해?", adapter=ADAPTER, draw_on=False))

## S12. 한계와 특이사항, 나만의 솔루션 기획, 결과 묶기

| 부품 | 잘 되는 것 | 한계 · 오늘 본 현상 | 고치는 방향 |
|---|---|---|---|
| LLM + Adapter | 말투 · 형식을 지시문 없이 | 문서를 붙이면 말투가 약해짐, 목표와 먼 질문에도 말투가 번짐 | 데이터에 문서 붙인 예시 섞기, 에폭 줄이기 |
| 검색(RAG) | 바뀌는 사실, 근거 보여 주기 | 다른 주제의 청크가 붙으면 그대로 틀림, 문서에 없으면 모름 | 청크 크기 · top_k · 키워드 비중, 문서 보강 |
| LoRA 로 지식 넣기 | 많이 되풀이하면 학습한 사실을 외움 | 덜 배우면 숫자를 지어내고, 많이 배우면 외운 사실 밖의 질문에 지어내며 관련 없는 질문에도 학습 문장의 모양이 번짐(9-1 ②) | 지식은 문서로(RAG) |
| 상태 JSON | 분위기 · 장면을 프로그램이 읽음 | 가끔 형식이 깨짐 | 예시 넣기, 다시 시키기, 형식 강제(5회차) |
| 그림 | 1초 안쪽, seed 로 같은 그림 | 손 · 글자 표현이 어색, 안전 필터 없음, 영어 묘사만 | 묘사 다듬기, 낱말 거르기 |
| 읽어 주기 | 빠르고 가벼움 | 숫자 · 영어를 못 읽음, 기계 소리 | 숫자 읽기, 큰 음성 모델 |
| 받아쓰기 | 한국어 · 영어 모두 | 고유 명사 · 숫자 실수(11/23 : 읽어 주기로 만든 소리에서 '망원경'을 '마음원경', '동아리방'을 '똥가리방'으로 들음), 잡음에 약함 | 큰 모델, 언어 지정, 받아쓴 글을 화면에 보여 고치게 함 |

**나만의 솔루션 기획(아래 셀에 적음)** : 누구를 위한 무엇인가, 어떤 부품을 어떤 순서로 쓰는가, 무엇을 학습시키고(Adapter) 무엇을 문서로 넣는가(RAG), 첫 번째 시험 질문 세 개.

In [ ]:
lab.note_panel()        # 기획과 오늘의 판단을 적음(보고서에 들어감)

In [ ]:
lab.stop_api()          # 띄운 API 를 끔
lab.pack()              # 결과 묶기(my/ 의 문서 · 설정 · Adapter 목록 · 그림책 · 기록)